<a href="https://colab.research.google.com/github/leman-cap13/NLP_projects/blob/main/An_Encoder_Decoder_Network_for_Neural_Machine_Translation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Hugging Face Pipelines

In [51]:
from transformers import pipeline

model_name = "distilbert-base-uncased-finetuned-sst-2-english"

classifier = pipeline(
    task="sentiment-analysis",
    model=model_name,
    truncation=True,
    max_length=512
)

texts = [
    "This was a great movie!",
    "This was not a great movie!",
    "The acting was amazing but the story was boring."
]

results = classifier(texts)

print(results)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.999869704246521}, {'label': 'NEGATIVE', 'score': 0.9998050332069397}, {'label': 'NEGATIVE', 'score': 0.997322142124176}]


In [52]:
classifier("I am from the USA")

[{'label': 'POSITIVE', 'score': 0.9642282128334045}]

In [53]:

classifier("I am from Iraq")

[{'label': 'NEGATIVE', 'score': 0.9706069231033325}]

In [54]:
ner = pipeline(
    "ner",
    aggregation_strategy="simple"
)

text = "Laman works as an AI Engineer in Baku and studies NLP with Hugging Face."

result = ner(text)

print(result)

[transformers] No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'entity_group': 'PER', 'score': np.float32(0.9974482), 'word': 'Laman', 'start': 0, 'end': 5}, {'entity_group': 'LOC', 'score': np.float32(0.99949443), 'word': 'Baku', 'start': 33, 'end': 37}, {'entity_group': 'MISC', 'score': np.float32(0.83429694), 'word': 'NL', 'start': 50, 'end': 52}, {'entity_group': 'PER', 'score': np.float32(0.40787792), 'word': 'Face', 'start': 67, 'end': 71}]


In [55]:
generator = pipeline(
    "text-generation",
    model="gpt2"
)

prompt = "In the future, artificial intelligence will"

result = generator(
    prompt,
    max_new_tokens=50,
    num_return_sequences=1
)

print(result[0]["generated_text"])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In the future, artificial intelligence will enable you to share your data with other people.

The future of technology is not going to be the end of the internet, but the beginning of a new era of communication.

If you want to build your own cloud service, be


In [56]:


fill_mask = pipeline(
    "fill-mask",
    model="bert-base-uncased"
)

text = "The capital of France is [MASK]."

result = fill_mask(text)

for item in result:
    print(item["sequence"], item["score"])

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


the capital of france is paris. 0.4167884886264801
the capital of france is lille. 0.07141659408807755
the capital of france is lyon. 0.0633925125002861
the capital of france is marseille. 0.044447433203458786
the capital of france is tours. 0.03029710426926613


In [57]:
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

texts = [
    "The GPU memory is not enough for training BERT.",
    "The football match ended with a dramatic goal.",
    "The recipe requires flour, eggs, and milk."
]

labels = ["machine learning", "sports", "food"]

for text in texts:
    result = classifier(text, candidate_labels=labels)
    print(text)
    print(result["labels"][0], result["scores"][0])
    print("-" * 80)

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

The GPU memory is not enough for training BERT.
machine learning 0.9694967269897461
--------------------------------------------------------------------------------
The football match ended with a dramatic goal.
sports 0.9957598447799683
--------------------------------------------------------------------------------
The recipe requires flour, eggs, and milk.
food 0.9880729913711548
--------------------------------------------------------------------------------


#An Encoder-Decoder Network for Neural Machine Translation

In [58]:
from datasets import load_dataset

nmt_original_valid_set, nmt_test_set = load_dataset(
    path="ageron/tatoeba_mt_train",
    name="eng-spa",
    split=["validation", "test"]
)

In [59]:
split = nmt_original_valid_set.train_test_split(train_size=0.8, seed=42)

nmt_train_set = split["train"]
nmt_valid_set = split["test"]

In [60]:
nmt_train_set[0]

{'source_text': 'Tom tried to break up the fight.',
 'target_text': 'Tom trató de disolver la pelea.',
 'source_lang': 'eng',
 'target_lang': 'spa'}

In [61]:
def train_eng_spa():
    for pair in nmt_train_set:
        yield pair["source_text"]
        yield pair["target_text"]

In [62]:
import tokenizers

max_length = 256
vocab_size = 10_000

nmt_tokenizer_model = tokenizers.models.BPE(unk_token="<unk>")
nmt_tokenizer = tokenizers.Tokenizer(nmt_tokenizer_model)

In [63]:
nmt_tokenizer.enable_padding(pad_id=0, pad_token="<pad>")
nmt_tokenizer.enable_truncation(max_length=max_length)

In [64]:
nmt_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()


In [65]:
nmt_tokenizer_trainer = tokenizers.trainers.BpeTrainer(
    vocab_size=vocab_size,
    special_tokens=[
        "<pad>",
        "<unk>",
        "<s>", # SOS
        "</s>" # EOS
    ]
)

In [66]:
nmt_tokenizer.train_from_iterator(
    train_eng_spa(),
    trainer=nmt_tokenizer_trainer
)

In [67]:
print(nmt_tokenizer.encode("I like soccer").ids)

[43, 401, 4381]


In [68]:

print(nmt_tokenizer.encode("<s> Me gusta el fútbol").ids)

[2, 396, 582, 219, 3356]


In [69]:
print(nmt_tokenizer.token_to_id("<pad>"))

0


In [70]:
print(nmt_tokenizer.token_to_id("<unk>"))

1


In [71]:
print(nmt_tokenizer.token_to_id("<s>"))

2


In [72]:
print(nmt_tokenizer.token_to_id("</s>"))

3


In [73]:
from collections import namedtuple

fields = [
    "src_token_ids", #eng
    "src_mask",
    "tgt_token_ids", #spa
    "tgt_mask"
]

class NmtPair(namedtuple("NmtPairBase", fields)):
    def to(self, device):
        return NmtPair(
            self.src_token_ids.to(device),
            self.src_mask.to(device),
            self.tgt_token_ids.to(device),
            self.tgt_mask.to(device)
        )

In [74]:
# src_token_ids → English token IDs
# src_mask      → English attention mask
# tgt_token_ids → Spanish decoder input token IDs
# tgt_mask      → Spanish attention mask

In [75]:
import torch
from torch.utils.data import DataLoader

In [76]:
def nmt_collate_fn(batch):
    src_texts = [pair["source_text"] for pair in batch ]

    tgt_texts = [f"<s> {pair['target_text']} </s>" for pair in batch ]

    src_encodings = nmt_tokenizer.encode_batch(src_texts)
    tgt_encodings = nmt_tokenizer.encode_batch(tgt_texts)

    src_token_ids = torch.tensor( [enc.ids for enc in src_encodings], dtype=torch.long )

    tgt_token_ids = torch.tensor([enc.ids for enc in tgt_encodings],  dtype=torch.long  )

    src_mask = torch.tensor([enc.attention_mask for enc in src_encodings], dtype=torch.long )

    tgt_mask = torch.tensor([enc.attention_mask for enc in tgt_encodings], dtype=torch.long)


    inputs = NmtPair(
        src_token_ids,
        src_mask,
        tgt_token_ids[:, :-1],  # [<s> i love movies </s>] => [<s> i love movies ]
        tgt_mask[:, :-1]
    )

    labels = tgt_token_ids[:, 1:] #[<s> i love movies </s>] => [ i love movies </s>]

    return inputs, labels

In [77]:
batch_size = 32

nmt_train_loader = DataLoader(
    nmt_train_set,
    batch_size=batch_size,
    collate_fn=nmt_collate_fn,
    shuffle=True
)

nmt_valid_loader = DataLoader(
    nmt_valid_set,
    batch_size=batch_size,
    collate_fn=nmt_collate_fn
)

nmt_test_loader = DataLoader(
    nmt_test_set,
    batch_size=batch_size,
    collate_fn=nmt_collate_fn
)

In [78]:
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence

In [79]:
class NmtModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim=512,
        pad_id=0,
        hidden_dim=512,
        n_layers=2
    ):
        super().__init__()

        self.embed = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_id
        )

        self.encoder = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True
        )

        self.decoder = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True
        )

        self.output = nn.Linear(
            in_features=hidden_dim,
            out_features=vocab_size
        )

    def forward(self, pair):
        src_embeddings = self.embed(pair.src_token_ids)
        tgt_embeddings = self.embed(pair.tgt_token_ids)

        src_lengths = pair.src_mask.sum(dim=1)

        src_packed = pack_padded_sequence(
            src_embeddings,
            lengths=src_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, hidden_states = self.encoder(src_packed)

        outputs, _ = self.decoder(
            tgt_embeddings,
            hidden_states
        )

        logits = self.output(outputs)

        return logits.permute(0, 2, 1) #(batch_size, tgt_len, vocab_size) but we need (batch_size, vocab_size, tgt_len)

In [80]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(42)

vocab_size = nmt_tokenizer.get_vocab_size()

nmt_model = NmtModel(vocab_size).to(device)

print(nmt_model)

NmtModel(
  (embed): Embedding(10000, 512, padding_idx=0)
  (encoder): GRU(512, 512, num_layers=2, batch_first=True)
  (decoder): GRU(512, 512, num_layers=2, batch_first=True)
  (output): Linear(in_features=512, out_features=10000, bias=True)
)


In [81]:
batch, labels = next(iter(nmt_train_loader))

batch = batch.to(device)
labels = labels.to(device)

logits = nmt_model(batch)

print("logits shape:", logits.shape)
print("labels shape:", labels.shape)

logits shape: torch.Size([32, 10000, 21])
labels shape: torch.Size([32, 21])


In [82]:
xentropy = nn.CrossEntropyLoss(ignore_index=0)

In [83]:
optimizer = torch.optim.NAdam(
    nmt_model.parameters(),
    lr=3e-4
)

In [84]:
def token_accuracy(logits, labels, pad_id=0):
    predictions = logits.argmax(dim=1)

    mask = labels != pad_id

    correct = (predictions == labels) & mask

    return correct.sum().float() / mask.sum().float()

In [85]:
from tqdm.auto import tqdm



def train_one_epoch(model, dataloader, optimizer, loss_fn, device, pad_id=0):
    model.train()

    total_loss = 0.0
    total_acc = 0.0
    total_batches = 0

    progress_bar = tqdm(dataloader, desc="Training")

    for batch, labels in progress_bar:
        batch = batch.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(batch)

        loss = loss_fn(logits, labels)
        acc = token_accuracy(logits, labels, pad_id=pad_id)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_acc += acc.item()
        total_batches += 1

        progress_bar.set_postfix({
            "loss": loss.item(),
            "acc": acc.item()
        })

    return total_loss / total_batches, total_acc / total_batches


def evaluate(model, dataloader, loss_fn, device, pad_id=0):
    model.eval()

    total_loss = 0.0
    total_acc = 0.0
    total_batches = 0

    progress_bar = tqdm(dataloader, desc="Validation")

    with torch.no_grad():
        for batch, labels in progress_bar:
            batch = batch.to(device)
            labels = labels.to(device)

            logits = model(batch)

            loss = loss_fn(logits, labels)
            acc = token_accuracy(logits, labels, pad_id=pad_id)

            total_loss += loss.item()
            total_acc += acc.item()
            total_batches += 1

            progress_bar.set_postfix({
                "val_loss": loss.item(),
                "val_acc": acc.item()
            })

    return total_loss / total_batches, total_acc / total_batches


num_epochs = 5

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(
        nmt_model,
        nmt_train_loader,
        optimizer,
        xentropy,
        device
    )

    valid_loss, valid_acc = evaluate(
        nmt_model,
        nmt_valid_loader,
        xentropy,
        device
    )

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Valid Loss: {valid_loss:.4f} | "
        f"Valid Acc: {valid_acc:.4f}"
    )


Training:   0%|          | 0/4933 [00:00<?, ?it/s]

Validation:   0%|          | 0/1234 [00:00<?, ?it/s]

Epoch 1/5 | Train Loss: 3.7581 | Train Acc: 0.4073 | Valid Loss: 2.9513 | Valid Acc: 0.4922


Training:   0%|          | 0/4933 [00:00<?, ?it/s]

Validation:   0%|          | 0/1234 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/1234 [00:00<?, ?it/s]

Epoch 4/5 | Train Loss: 1.5635 | Train Acc: 0.6836 | Valid Loss: 2.0148 | Valid Acc: 0.6213


Training:   0%|          | 0/4933 [00:00<?, ?it/s]

Validation:   0%|          | 0/1234 [00:00<?, ?it/s]

Epoch 5/5 | Train Loss: 1.3014 | Train Acc: 0.7262 | Valid Loss: 1.9829 | Valid Acc: 0.6270


In [86]:
def translate(
    model,
    src_text,
    max_length=20,
    pad_id=0,
    eos_id=3
):
    model.eval()

    tgt_text = ""

    for index in range(max_length):

        batch, _ = nmt_collate_fn([
            {
                "source_text": src_text,
                "target_text": tgt_text
            }
        ])

        batch = batch.to(device)

        with torch.no_grad():
            y_logits = model(batch)

        y_token_ids = y_logits.argmax(dim=1)

        next_token_id = y_token_ids[0, index].item()

        next_token = nmt_tokenizer.id_to_token(next_token_id)

        tgt_text += " " + next_token

        if next_token_id == eos_id:
            break

    return tgt_text




In [87]:
nmt_model.eval()


NmtModel(
  (embed): Embedding(10000, 512, padding_idx=0)
  (encoder): GRU(512, 512, num_layers=2, batch_first=True)
  (decoder): GRU(512, 512, num_layers=2, batch_first=True)
  (output): Linear(in_features=512, out_features=10000, bias=True)
)

In [88]:
translate(
    nmt_model,
    "I like to play soccer with my friends."
)

' Me gusta jugar con los amigos de fútbol . </s>'

In [89]:
translate(
    nmt_model,
    "Hi , my friend"
)


' Hola , amigo mío . </s>'

#Beam Search

In [90]:
import torch.nn.functional as F

In [91]:
def make_nmt_pair_for_generation(src_text, tgt_token_ids, device):
    src_encoding = nmt_tokenizer.encode(src_text)

    src_token_ids = torch.tensor(
        [src_encoding.ids],
        dtype=torch.long,
        device=device
    )

    src_mask = torch.tensor(
        [src_encoding.attention_mask],
        dtype=torch.long,
        device=device
    )

    tgt_token_ids = torch.tensor(
        [tgt_token_ids],
        dtype=torch.long,
        device=device
    )

    tgt_mask = torch.ones_like(tgt_token_ids)

    return NmtPair(
        src_token_ids=src_token_ids,
        src_mask=src_mask,
        tgt_token_ids=tgt_token_ids,
        tgt_mask=tgt_mask
    )

In [101]:
def beam_search_translate(
    model,
    src_text,
    beam_width=3,
    max_length=20,
    device=device
):
    model.eval()

    sos_id = nmt_tokenizer.token_to_id("<s>")
    eos_id = nmt_tokenizer.token_to_id("</s>")


    beams = [
        ([sos_id], 0.0, False)
    ]

    with torch.no_grad():
        for step in range(max_length):
            all_candidates = []

            for token_ids, score, finished in beams:


                if finished:
                    all_candidates.append(
                        (token_ids, score, finished)
                    )
                    continue

                pair = make_nmt_pair_for_generation(
                    src_text=src_text,
                    tgt_token_ids=token_ids,
                    device=device
                )

                logits = model(pair)

                next_token_logits = logits[0, :, -1]

                log_probs = F.log_softmax(
                    next_token_logits,
                    dim=-1
                )


                top_log_probs, top_token_ids = torch.topk(
                    log_probs,
                    k=beam_width
                )

                for log_prob, next_token_id in zip(top_log_probs, top_token_ids):
                    next_token_id = next_token_id.item()

                    new_token_ids = token_ids + [next_token_id]
                    new_score = score + log_prob.item()
                    new_finished = next_token_id == eos_id

                    all_candidates.append(
                        (new_token_ids, new_score, new_finished)
                    )

            all_candidates = sorted(
                all_candidates,
                key=lambda x: x[1],
                reverse=True
            )

            beams = all_candidates[:beam_width]

            if all(finished for _, _, finished in beams):
                break

    best_token_ids, best_score, finished = beams[0]

    best_token_ids = best_token_ids[1:]

    if eos_id in best_token_ids:
        eos_index = best_token_ids.index(eos_id)
        best_token_ids = best_token_ids[:eos_index]

    translation = nmt_tokenizer.decode(
        best_token_ids,
        skip_special_tokens=True
    )

    return translation

In [102]:
beam_search_translate(
    nmt_model,
    "I like to play soccer with my friends.",
    beam_width=3,
    max_length=20
)

'pity pity pity respect respect quedes ora ora puse puse enseñ wai suit suit suit sigu sigu gia ests gia'

#Attention Mechanisms

In [104]:
from torch.nn.utils.rnn import pad_packed_sequence, pack_padded_sequence

In [105]:
def attention(query, key, value):
    scores = query @ key.transpose(1, 2)
    weights = torch.softmax(scores, dim=-1)
    return weights @ value

In [106]:
class NmtAttentionModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim=512,
        pad_id=0,
        hidden_dim=512,
        n_layers=2
    ):
        super().__init__()

        self.embed = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_id
        )

        self.encoder = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True
        )

        self.decoder = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True
        )

        self.output = nn.Linear(
            in_features=2 * hidden_dim,
            out_features=vocab_size
        )

    def forward(self, pair):
        src_embeddings = self.embed(pair.src_token_ids)
        tgt_embeddings = self.embed(pair.tgt_token_ids)

        src_lengths = pair.src_mask.sum(dim=1)

        src_packed = pack_padded_sequence(
            src_embeddings,
            lengths=src_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        encoder_outputs_packed, hidden_states = self.encoder(src_packed)

        encoder_outputs, _ = pad_packed_sequence(
            encoder_outputs_packed,
            batch_first=True
        )

        decoder_outputs, _ = self.decoder(
            tgt_embeddings,
            hidden_states
        )

        attn_output = attention(
            query=decoder_outputs,
            key=encoder_outputs,
            value=encoder_outputs
        )

        combined_output = torch.cat(
            (attn_output, decoder_outputs),
            dim=-1
        )

        logits = self.output(combined_output)

        return logits.permute(0, 2, 1)

In [107]:
nmt_model = NmtModel(vocab_size).to(device)

In [108]:
nmt_attn_model = NmtAttentionModel(vocab_size).to(device)

In [109]:
optimizer = torch.optim.NAdam(
    nmt_attn_model.parameters(),
    lr=3e-4
)

In [110]:
xentropy = nn.CrossEntropyLoss(ignore_index=0)

In [ ]:
num_epochs = 5

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(
        nmt_attn_model,
        nmt_train_loader,
        optimizer,
        xentropy,
        device
    )

    valid_loss, valid_acc = evaluate(
        nmt_attn_model,
        nmt_valid_loader,
        xentropy,
        device
    )

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Valid Loss: {valid_loss:.4f} | "
        f"Valid Acc: {valid_acc:.4f}"
    )

Training:   0%|          | 0/4933 [00:00<?, ?it/s]

In [ ]:
translate(
    nmt_attn_model,
    "I like to play soccer with my friends."
)

In [ ]:
beam_search_translate(
    nmt_attn_model,
    "I like to play soccer with my friends.",
    beam_width=3,
    max_length=20
)